In [2]:
%pip install matplotlib

In [3]:
import subprocess
import os
import matplotlib.pyplot as plt

def sh(command):
    res = subprocess.run(["bash", "-c", command], 
                       capture_output=True, 
                       text=True)
    if res.returncode != 0:
        print(f"Error executing command: {command}")
        print(f"Return code: {res.returncode}")
        print(f"stderr: {res.stderr}")
        raise Exception("Command failed")
    
    return res.stdout

The Code in this file should be used to scrape and set up the training data, compile relevant CLIs and so forth.

In [4]:
# Install CLI Tools required later.
sh("cd chord-simplify && ./install.sh")
sh("cd frequency-analysis && ./install.sh")


This next one can really take a while, and scraping the data is somewhat expensive for ultimate guitar probably.

Please uncomment this, IFF you know what you're doing and actually plan on using this.

In [5]:
# Scrape chordsheets from Ultimate-Guitar

if False:
    subprocess.run(["bash", "-c", "cd ug-scrape && node main.js"])
else:
    print("Skipping UG scrape, assuming data is already present.")

In [6]:
# Let's simplify the whitespace in all the files we just scraped.
sh("""
    for f in ug-scrape/output/*; do
        ./simplify-whitespace "$f" &
    done
   """)

## Generating Set of Raw tokens

We do not generate the final list of tokens immediately.
This is because the amount of final tokens is immense, and also some outliers are still present.

We use the set of raw tokens for further analysis down the line.

In [7]:
# Create directory for first stage of token extraction
if not os.path.exists("output"):
    os.mkdir("output")

In [8]:
subprocess.run(["bash", "-c", """
    npx tsx token-extract/main.ts ug-scrape/output ./config/almost-all-valid-tokens.csv > output/raw-tokens
"""])

In [9]:
# you probably won't need this, here I'm using other data local to my machine as well.

if os.path.exists("token-extract/ultimate-guitar"):
    subprocess.run(["bash", "-c", """
        cd token-extract
        npx tsx main.js :default ../config/almost-all-valid-tokens.csv  >> ../output/raw-tokens
    """])

In [10]:
# This step is about analysing the distribution of our tokens / training data and refining the actual set of tokens we use for our model.

rawTokenAnalysis = sh("frequency-analysis --no-header < output/raw-tokens")
print(rawTokenAnalysis)
chordFrequency = list()
chordToken = list()
for line in rawTokenAnalysis.splitlines():
    freq, token = line.split(";")
    chordFrequency.append(int(freq))
    chordToken.append(token)

In [16]:

# We're drawing a graph of chordFrequency to help decide a cutoff point.
fig, ax = plt.subplots()
ax.plot(chordFrequency)
ax.fill_between(range(len(chordFrequency)), chordFrequency, color='skyblue', alpha=0.5)
ax.set_ylim(top=10000, bottom=0)
ax.set_xlim(left=0, right=512)
ax.set_ylabel('Frequency')
ax.set_xlabel('Token rank')
ax.set_title('Token Frequency Distribution')
ax.grid(True)
#fig.savefig('token_frequency_distribution.png')
plt.show()

In [17]:
# This variable is critical for our LLM infrastructure. 
# It defines the size of the vocabulary we will use and as such impacts the Models architecture.
# This number was chosen based on the output of our frequency analysis; 
# after rank 200 we reach barely a hundred occurencec per token, at 265 it's almost half that. Further down the line I suspect it's too little to work with.
# At the same time, we still cover plenty of interesting chords and by far all "standard" chords this way.
VOCABULARY_SIZE = 256

validChords = chordToken[:VOCABULARY_SIZE]
# write validChords to file
with open("config/valid-tokens.csv", "w") as f:
    for chord in validChords:
        f.write(f"{chord}\n")

print(" ".join(validChords))

# Next step is to generate the final input data.

We take the total amount of tokens generated and 
    - 1.) try to simplify chords to fit our target set of valid chords (e.g. Emaj7 => E7)  
    - 2.) Remove any tokens that don't fit this pattern, as we still have some undected noise here.
  
After this step there is no filtering out of the noise feasable anymore

In [13]:
sh("chord-simplify config/valid-tokens.csv < output/raw-tokens > output/tokens")
# you can open the files in a diff viewer to see the difference.
print("final input data for llm training now lives in output/tokens")

In [14]:
# here we define our mapper functions to go from token to number and back again.
def tokenToNumber(token: str) -> int:
    return validChords.index(token)

def numberToToken(number: int) -> str:
    return validChords[number]